In [20]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, 
    classification_report, 
    ConfusionMatrixDisplay, 
    mean_squared_error, 
    make_scorer, mean_squared_error,  
    mean_absolute_error, r2_score
)
from xgboost import XGBRegressor

In [30]:
train_df = pd.read_csv("combined_train_with_labels.csv.gz")
# npml_file = pd.read_csv("combined_npmled_features.csv.gz")

In [22]:
eunice = pd.read_csv("../eunice_csv_files/eunice_combined_npml.csv.gz")
nomin = pd.read_csv("../nomin_csv_files/combined_npml_n.csv.gz")
prithvi = pd.read_csv("../prithvi_csv_files/prithvi_combined_npml.csv.gz")
# jade = pd.read_csv("../jade_csv_files/jade_combined_npml.csv.gz")

print(eunice.shape)
print(nomin.shape)
print(prithvi.shape)

(159697, 8)
(159697, 11)
(159697, 9)


In [23]:
eunice.columns
eunice['id'][0]

'3033789_npml_0'

In [24]:
nomin.columns
nomin['id'][2]

'3033791_npml_0'

In [25]:
prithvi.columns
prithvi['id'][0]

'3033789_npml_0'

In [26]:
eunice = eunice.drop(columns=["file"], errors="ignore")
nomin = nomin.drop(columns=["file"], errors="ignore")
prithvi = prithvi.drop(columns=["file"], errors="ignore")
# jade = jade.drop(columns=["file"], errors="ignore")

In [27]:
eunice["id"] = eunice["id"].astype(str)
nomin["id"] = nomin["id"].astype(str)
prithvi["id"] = prithvi["id"].astype(str)
# jade["id"] = jade["id"].astype(str)

In [28]:
df = eunice.merge(nomin, on="id")
df = df.merge(prithvi, on="id")

# Jade merging
# df = df.merge(jade, on="id")

print(df.shape)

(159697, 24)


In [29]:
df

,id,ED,HWP,ND80,PPR,SCA,LQ80,tail_slope_no_pz,current_kurtosis,time_to_peak,...,spectral_centroid_power,current_width,tp0,tdrift10,tdrift50,tdrift99,tfr,peak_count,gbn,bpr
0,3033789_npml_0,3405,2090.0,0.0,0.694995,0.034962,-3.234871e+05,-10061.013124,6.674751,83,...,104.055619,0.159431,968,17.0,28.0,83.0,0.153772,5,1.161357,0.055796
1,3033790_npml_0,3406,2165.0,0.0,0.702777,0.035541,-2.097532e+05,-9089.512080,10.563529,180,...,107.098676,0.121488,942,0.0,71.0,180.0,0.142348,3,1.542807,0.062473
2,3033791_npml_0,3408,2129.0,0.0,0.699074,0.035238,-2.425070e+05,-9449.901755,5.231941,185,...,103.335168,0.182753,940,0.0,76.0,185.0,0.144746,6,1.171004,0.049921
3,3033792_npml_0,3411,2125.0,0.0,0.701728,0.035928,-2.223822e+05,-8567.556832,7.421443,201,...,106.077102,0.154580,936,0.0,101.0,201.0,0.135823,10,2.442536,0.053420
4,3033793_npml_0,3405,2002.0,0.0,0.684008,0.035634,-2.887550e+05,-10081.558061,8.552893,96,...,109.909804,0.143348,952,27.0,66.0,96.0,0.155597,24,1.367881,0.061533
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159692,3193481_npml_2,3408,2046.0,0.0,0.693882,0.034458,-8.314132e+05,-10371.604799,7.796771,77,...,108.444858,0.148676,972,26.0,54.0,77.0,0.149046,2,2.085564,0.059918
159693,3193482_npml_2,3411,2060.0,0.0,0.696157,0.034467,-1.789650e+06,-10700.698164,10.240402,94,...,111.463167,0.122861,967,54.0,76.0,93.0,0.143292,1,1.029563,0.062920
159694,3193483_npml_2,3409,2082.0,0.0,0.697220,0.034344,-1.315494e+06,-10687.382341,10.704435,86,...,109.771213,0.121086,966,43.0,60.0,85.0,0.146062,1,1.433963,0.064889
159695,3193484_npml_2,3405,2062.0,0.0,0.693403,0.034225,-2.350121e+06,-10623.611465,8.239963,65,...,106.924402,0.136744,981,15.0,28.0,65.0,0.154574,1,1.453343,0.060919


## everyone include your classification model here so once our final npml file is done, we can just create the predictions csv in here

### high_avse classification

In [ ]:
high_avse_xgb_threshold = 0.7

high_avse_xgb = xgb.XGBClassifier(
    subsample=1.0,
    scale_pos_weight=0.5,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.3,
    gamma=0.1,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

high_avse_xgb.fit(X_train, y_train)

y_probs_high_avse = high_avse_xgb.predict_proba(X_test)[:, 1]
y_pred_high_avse = (y_probs_high_avse >= high_avse_xgb_threshold).astype(int)

### lq classification

In [ ]:
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    ))
])

In [ ]:
df_train = pd.read_csv("combined_train_with_labels.csv.gz")

drop_cols = [
    "id",
    "energy_label",
    "psd_label_low_avse",
    "psd_label_high_avse",
    "psd_label_dcr",
    "psd_label_lq"
]

X = df_train.drop(columns=drop_cols)
y = df_train["psd_label_lq"]

rf_pipe.fit(X, y)

In [ ]:
X_npml = df.drop(columns=["id"], errors="ignore")

X_npml = X_npml[X.columns]

scores = rf_pipe.predict_proba(X_npml)[:, 1]
best_threshold = 0.42

pred_lq = (scores >= best_threshold).astype(int)

In [ ]:
pred_df = pd.DataFrame({
    "id": df["id"],
    "predicted_label_lq": pred_lq
})

pred_df.to_csv("npml_lq_predictions.csv", index=False)

# Low avse classification - Random Forest

In [31]:
from sklearn.model_selection import train_test_split
target = "psd_label_low_avse"
drop_cols = ["psd_label_low_avse","psd_label_high_avse","psd_label_dcr","psd_label_lq","energy_label","id"]
X = train_df.drop(columns=drop_cols)
y = train_df[target]
X = X.fillna(X.median())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [33]:
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# LightGBM Regression

In [34]:
import lightgbm as lgb
import numpy as np
target = "energy_label"
drop_cols = ["psd_label_low_avse","psd_label_high_avse","psd_label_dcr","psd_label_lq","energy_label","id"]
X = train_df.drop(columns=drop_cols)
y = train_df[target]
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)

In [35]:
lgb_model = lgb.LGBMRegressor(learning_rate=0.03,n_estimators=5000,num_leaves=63,subsample=0.8,colsample_bytree=0.8,random_state=42,
    n_jobs=-1)
lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026905 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5335
[LightGBM] [Info] Number of data points in the train set: 832000, number of used features: 23
[LightGBM] [Info] Start training from score 638.761951


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [36]:
y_train_pred = lgb_model.predict(X_train)
y_test_pred = lgb_model.predict(X_test)